In [ ]:
import pandas as pd

In [ ]:
# using nrows=100 for ease during iteration
#df_train_labels = pd.read_csv("../data/train_labels.csv", nrows=100)
df_train_labels = pd.read_csv("../data/train_labels.csv")

In [ ]:
#creating new cols
df_train_labels[["version_standard","chemical_formula","remainder"]] = df_train_labels["InChI"].str.split("/", n=2, expand=True)

In [ ]:
df_train_labels.head(10)

Are they all version 1 and standard? (spoiler - yes)

In [ ]:
df_train_labels['version_standard'].describe()

### Understanding the elements

Now we have a breakdown of all the molecules, including a breakdown of their elemental composition, we can do some analysis.  For example, how many different elements are present and in what quantities?

We can write a simple regex to count the elements given a formula of the form `AiBjCdk`, where `A, B, Cd` are elements, and `i,j,k` their counts.  The regex looks like `([A-Z][a-z]?)(\d*)` and can be broken down as follows:

- `[A-Z]` matches a capital letter, so `A` `B` and `C` in our example.
- `[a-z]?` optionally matches a lower case (0 or 1 occurance), will match the `d` in our case.  Elements are only ever an uppercase letter followed by zero or one lowercase letters.
- `(\d)` captures any digit after the number, and the `*` catches 0 or more.  If it finds zero, we can replace it with a one since single element counts in a molecule do not get given an explicit `1` count.

In [ ]:
import re

def parse_formula(formula):
    parts = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    return {
        element: int(count) if count else 1
        for element, count in parts
    }

print(parse_formula("C13H15BrN2O3"))


parts = re.findall(r'([A-Z][a-z]?)(\d*)', "C13H15BrN2O3")
print(parts)

We could capture it in a formula like this, but we will use the vectorised way of doing it. `re.findall` is vectorised, so applying it to the whole column, rather than row by row will be much faster


In [ ]:
df_parts = pd.DataFrame()
df_parts["parts"] =  df_train_labels["chemical_formula"].str.findall(r'([A-Z][a-z]?)(\d*)')

In [ ]:
df_parts["parts"]

`explode` will decompose the dataframe into one long series. It reshapes data without python loops so will be fast.

In [ ]:
df_exploded = df_parts.explode("parts")
df_exploded

We can separate the elements out and work out a count for each

In [ ]:
df_exploded[["element", "count"]] = pd.DataFrame(
    df_exploded["parts"].tolist(), index=df_exploded.index
)
df_exploded["count"] = df_exploded["count"].replace("", "1").astype(int)
df_exploded

Finally we can create a pivot table to summarise what we have on a molecule by molecule (row by row) basis.

In [ ]:
element_counts = df_exploded.pivot_table(
    index=df_exploded.index,
    columns="element",
    values="count",
    aggfunc="sum",
    fill_value=0
)
element_counts

In [ ]:
f"Total atom count: {element_counts.sum().sum():,}"

In [ ]:
import matplotlib.pyplot as plt

element_counts.sum().div(1e6).sort_values(ascending=False).plot(kind="bar")
plt.title("Total atom counts")
plt.xlabel("Element")
plt.ylabel("Count (Millions)")
plt.show()

As expected, H, C, N and O dominate the counts. This is organic chemistry!

We avoided using `apply` to use our regex to expand into columns - it is slowwww as it uses python loops
`df_speed_test = df_train_labels["chemical_formula"].apply(parse_formula).apply(pd.Series)`

We have over 2.4 million molecules (rows) and 110 Million atoms so this would take a long time to complete.

In [ ]:
52,283,920